# Edit wget scripts

Kernel: e.g. `postprocess`

Edit wget scripts by removing files for years that are not of interest from the list of files and by splitting the scripts into scripts downloading at most 1000 files (this is the maximum). Careful: some server automatically cut off at 1000 files, without warning. So in this case the creation of wgets should be limited to 1000 files.

In [1]:
import os

In [2]:
def remove_datasets(path_wget: str, to_remove: list, verbose: bool = False):
    """ Edit a wget script by removing datasets that contain a certain string from a list
    path_wget: the path of the wget downloading script
    to_remove: list of strings, if the filename contains one of these the file is removed
    """
    with open(path_wget, 'r') as file:
        lines = file.readlines()
        lines_reduced = [f for f in lines if not any(year in f for year in to_remove)] # Select the lines we want to keep
        print(f"Original number of files: {len([l for l in lines if '.nc' in l])}")
        print(f"Number of files left: {len([l for l in lines_reduced if '.nc' in l])}")
        if verbose:
            print('Removed files:')
            print(*[f for f in lines if any(year in f for year in to_remove)])

    with open(path_wget, 'w') as file:
        file.writelines(lines_reduced)

def split_wget_script(path: str):
    """ Split a wget script into several scripts of at most 1000 lines, since this is the maximum number of lines for wget
    path: the path to the wget script
    """
    with open(path, 'r') as file: 
        # Read the full file
        lines = file.readlines()
        files = [l for l in lines if '.nc' in l]
        path_file1 = path

        # If the script did not have more than 1000 files to begin with, just exit the function
        if len(files) < 1000: return

        # Check if the number of files is greater than 1000: a wget script can download maximum 1000 files
        while len(files)>1000:
            # Construct a file with the first 999 filenames to download
            start = lines.index(files[0])
            end = lines.index(files[-1])
            lines_file1 = lines[0: start] + files[0:999] + lines[end+1: len(lines)]

            # Construct the new filename for the file that is split off
            if "part_" in path_file1:
                index = path_file1.split('part_')[-1].split('.')[0]
                path_file1 = path_file1.replace(index, str(int(index)+1))
            else: path_file1 = path_file1.replace('.sh', '_part_1.sh')

            # Save this new file
            with open(path_file1,'w') as file1:
                file1.writelines(lines_file1)
                print(f'Saved {path_file1}')
            print(f"Number of files in {path_file1}: {len([l for l in lines_file1 if '.nc' in l])}")
            
            # Reinitialise the lines with the remaining filenames
            lines = lines[0: start] + files[999:]+ lines[end+1: len(lines)]
            files = [l for l in lines if '.nc' in l]
        
        # Save the remaining lines
        index = path_file1.split('part_')[-1].split('.')[0]
        with open(path_file1.replace(index, str(int(index)+1)),'w') as file1:
            file1.writelines(lines)
            print(f'Saved {path_file1.replace(index, str(int(index)+1))}')
            print(f"Number of files in {path_file1.replace(index, str(int(index)+1))}: {len([l for l in lines if '.nc' in l])}")

        # Remove the original file
        os.remove(path)

def prepare_wget(path: str):
    """ Prepare a wget script byt removing unnecessary files and splitting into scripts of at most 1000 files
    path: the path to the wget script
    """
    to_remove = [
            '_1950', '_1951', '_1952', '_1953', '_1954', '_1955', '_1956', '_1957', '_1958', '_1959', 
            '_1960', '_1961', '_1962', '_1963', '_1964', '_1965', '_1966', '_1967', '_1968', '_1969',
            '_2051', '_2052', '_2053', '_2054', '_2055', '_2056', '_2057', '_2058', '_2059', 
            '_2060', '_2061', '_2062', '_2063', '_2064', '_2065', '_2066', '_2067', '_2068', '_2069',
            '_2070', '_2071', '_2072', '_2073', '_2074', '_2075', '_2076', '_2077', '_2078', '_2079',
            '_2080', '_2081', '_2082', '_2083', '_2084', '_2085', '_2086', '_2087', '_2088', '_2089',
            '_2090', '_2091', '_2092', '_2093', '_2094', '_2095', '_2096', '_2097', '_2098', '_2099',
            ]
    remove_datasets(path_wget = path, to_remove = to_remove)
    split_wget_script(path)

In [3]:
prepare_wget('path_to_wget/wget.sh') #TODO EDIT PATH TO WGET SCRIPT

Original number of files: 299
Number of files left: 219
